In [ ]:
!pip install catboost

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# Step 1: Load the Base Models (DenseNet121 and EfficientNetB0 for Leaf)
leaf_model_1_path = '/content/drive/MyDrive/Capstone Models/DenseNet121_leaf.keras'
leaf_model_2_path = '/content/drive/MyDrive/Capstone Models/EfficientNetB0_leaf.keras'

leaf_model_1 = load_model(leaf_model_1_path)
leaf_model_2 = load_model(leaf_model_2_path)

In [ ]:
# Step 2: Load the Dataset (Leaf Test Dataset)
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

test_dir = "/content/drive/MyDrive/Capstone Dataset/Guava_Leaf_Dieases/Test"

# Define leaf classes manually
leaf_classes = ['Anthracnose', 'Canker', 'Dot', 'Healthy', 'Rust']  # Replace with actual leaf classes

# Load the test dataset
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

Found 1150 files belonging to 5 classes.


In [ ]:
# Step 3: Generate Predictions from Base Models
y_prob_leaf_model_1 = leaf_model_1.predict(test_ds)  # Predictions from DenseNet121
y_prob_leaf_model_2 = leaf_model_2.predict(test_ds)  # Predictions from EfficientNetB0

# Step 4: Stack the Predictions (Create Meta-Features)
X_meta_leaf = np.column_stack((
    y_prob_leaf_model_1,
    y_prob_leaf_model_2
))

# True labels for the test set (adjust this to match your dataset)
y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)

36/36 ━━━━━━━━━━━━━━━━━━━━ 232s 6s/step
36/36 ━━━━━━━━━━━━━━━━━━━━ 14s 279ms/step


# Step 5: Define and Train Meta-Models (Each Section)

In [ ]:
## 1. Logistic Regression
print("\nTraining Logistic Regression...")
meta_model_lr = LogisticRegression()
meta_model_lr.fit(X_meta_leaf, y_true)
y_meta_pred_lr = meta_model_lr.predict(X_meta_leaf)
print(f"Logistic Regression Accuracy: {accuracy_score(y_true, y_meta_pred_lr):.4f}")
print(classification_report(y_true, y_meta_pred_lr, target_names=leaf_classes))


Training Logistic Regression...
Logistic Regression Accuracy: 0.8113
              precision    recall  f1-score   support

 Anthracnose       0.78      0.91      0.84       230
      Canker       0.89      0.87      0.88       230
         Dot       0.65      0.58      0.61       230
     Healthy       0.81      0.91      0.86       230
        Rust       0.93      0.79      0.85       230

    accuracy                           0.81      1150
   macro avg       0.81      0.81      0.81      1150
weighted avg       0.81      0.81      0.81      1150



In [ ]:
## 2. Random Forest
print("\nTraining Random Forest...")
meta_model_rf = RandomForestClassifier()
meta_model_rf.fit(X_meta_leaf, y_true)
y_meta_pred_rf = meta_model_rf.predict(X_meta_leaf)
print(f"Random Forest Accuracy: {accuracy_score(y_true, y_meta_pred_rf):.4f}")
print(classification_report(y_true, y_meta_pred_rf, target_names=leaf_classes))


Training Random Forest...
Random Forest Accuracy: 1.0000
              precision    recall  f1-score   support

 Anthracnose       1.00      1.00      1.00       230
      Canker       1.00      1.00      1.00       230
         Dot       1.00      1.00      1.00       230
     Healthy       1.00      1.00      1.00       230
        Rust       1.00      1.00      1.00       230

    accuracy                           1.00      1150
   macro avg       1.00      1.00      1.00      1150
weighted avg       1.00      1.00      1.00      1150



In [ ]:
## 3. Gradient Boosting
print("\nTraining Gradient Boosting...")
meta_model_gb = GradientBoostingClassifier()
meta_model_gb.fit(X_meta_leaf, y_true)
y_meta_pred_gb = meta_model_gb.predict(X_meta_leaf)
print(f"Gradient Boosting Accuracy: {accuracy_score(y_true, y_meta_pred_gb):.4f}")
print(classification_report(y_true, y_meta_pred_gb, target_names=leaf_classes))


Training Gradient Boosting...
Gradient Boosting Accuracy: 1.0000
              precision    recall  f1-score   support

 Anthracnose       1.00      1.00      1.00       230
      Canker       1.00      1.00      1.00       230
         Dot       1.00      1.00      1.00       230
     Healthy       1.00      1.00      1.00       230
        Rust       1.00      1.00      1.00       230

    accuracy                           1.00      1150
   macro avg       1.00      1.00      1.00      1150
weighted avg       1.00      1.00      1.00      1150



In [ ]:
## 4. Support Vector Classifier (SVC)
print("\nTraining Support Vector Classifier (SVC)...")
meta_model_svc = SVC()
meta_model_svc.fit(X_meta_leaf, y_true)
y_meta_pred_svc = meta_model_svc.predict(X_meta_leaf)
print(f"SVC Accuracy: {accuracy_score(y_true, y_meta_pred_svc):.4f}")
print(classification_report(y_true, y_meta_pred_svc, target_names=leaf_classes))


Training Support Vector Classifier (SVC)...
SVC Accuracy: 0.8443
              precision    recall  f1-score   support

 Anthracnose       0.81      0.93      0.87       230
      Canker       0.90      0.88      0.89       230
         Dot       0.75      0.63      0.69       230
     Healthy       0.83      0.93      0.87       230
        Rust       0.95      0.84      0.89       230

    accuracy                           0.84      1150
   macro avg       0.85      0.84      0.84      1150
weighted avg       0.85      0.84      0.84      1150



In [ ]:
## 5. Multi-layer Perceptron (MLP)
print("\nTraining Multi-layer Perceptron (MLP)...")
meta_model_mlp = MLPClassifier()
meta_model_mlp.fit(X_meta_leaf, y_true)
y_meta_pred_mlp = meta_model_mlp.predict(X_meta_leaf)
print(f"MLP Accuracy: {accuracy_score(y_true, y_meta_pred_mlp):.4f}")
print(classification_report(y_true, y_meta_pred_mlp, target_names=leaf_classes))


Training Multi-layer Perceptron (MLP)...
MLP Accuracy: 0.8513
              precision    recall  f1-score   support

 Anthracnose       0.83      0.94      0.88       230
      Canker       0.91      0.89      0.90       230
         Dot       0.75      0.62      0.68       230
     Healthy       0.83      0.93      0.87       230
        Rust       0.94      0.88      0.91       230

    accuracy                           0.85      1150
   macro avg       0.85      0.85      0.85      1150
weighted avg       0.85      0.85      0.85      1150



/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
## 6. AdaBoost Classifier
print("\nTraining AdaBoost Classifier...")
meta_model_ada = AdaBoostClassifier()
meta_model_ada.fit(X_meta_leaf, y_true)
y_meta_pred_ada = meta_model_ada.predict(X_meta_leaf)
print(f"AdaBoost Accuracy: {accuracy_score(y_true, y_meta_pred_ada):.4f}")
print(classification_report(y_true, y_meta_pred_ada, target_names=leaf_classes))


Training AdaBoost Classifier...
AdaBoost Accuracy: 0.8096
              precision    recall  f1-score   support

 Anthracnose       0.77      0.93      0.84       230
      Canker       0.82      0.86      0.84       230
         Dot       0.77      0.54      0.63       230
     Healthy       0.82      0.93      0.87       230
        Rust       0.87      0.80      0.83       230

    accuracy                           0.81      1150
   macro avg       0.81      0.81      0.80      1150
weighted avg       0.81      0.81      0.80      1150



In [ ]:
## 7. Decision Tree Classifier
print("\nTraining Decision Tree Classifier...")
meta_model_dt = DecisionTreeClassifier()
meta_model_dt.fit(X_meta_leaf, y_true)
y_meta_pred_dt = meta_model_dt.predict(X_meta_leaf)
print(f"Decision Tree Accuracy: {accuracy_score(y_true, y_meta_pred_dt):.4f}")
print(classification_report(y_true, y_meta_pred_dt, target_names=leaf_classes))


Training Decision Tree Classifier...
Decision Tree Accuracy: 1.0000
              precision    recall  f1-score   support

 Anthracnose       1.00      1.00      1.00       230
      Canker       1.00      1.00      1.00       230
         Dot       1.00      1.00      1.00       230
     Healthy       1.00      1.00      1.00       230
        Rust       1.00      1.00      1.00       230

    accuracy                           1.00      1150
   macro avg       1.00      1.00      1.00      1150
weighted avg       1.00      1.00      1.00      1150



In [ ]:
## 8. K-Nearest Neighbors (KNN)
print("\nTraining K-Nearest Neighbors (KNN)...")
meta_model_knn = KNeighborsClassifier()
meta_model_knn.fit(X_meta_leaf, y_true)
y_meta_pred_knn = meta_model_knn.predict(X_meta_leaf)
print(f"KNN Accuracy: {accuracy_score(y_true, y_meta_pred_knn):.4f}")
print(classification_report(y_true, y_meta_pred_knn, target_names=leaf_classes))


Training K-Nearest Neighbors (KNN)...
KNN Accuracy: 0.9157
              precision    recall  f1-score   support

 Anthracnose       0.92      0.95      0.93       230
      Canker       0.92      0.92      0.92       230
         Dot       0.87      0.84      0.85       230
     Healthy       0.90      0.95      0.92       230
        Rust       0.98      0.92      0.95       230

    accuracy                           0.92      1150
   macro avg       0.92      0.92      0.92      1150
weighted avg       0.92      0.92      0.92      1150



In [ ]:
## 9. Gaussian Naive Bayes (GNB)
print("\nTraining Gaussian Naive Bayes (GNB)...")
meta_model_gnb = GaussianNB()
meta_model_gnb.fit(X_meta_leaf, y_true)
y_meta_pred_gnb = meta_model_gnb.predict(X_meta_leaf)
print(f"GNB Accuracy: {accuracy_score(y_true, y_meta_pred_gnb):.4f}")
print(classification_report(y_true, y_meta_pred_gnb, target_names=leaf_classes))


Training Gaussian Naive Bayes (GNB)...
GNB Accuracy: 0.7843
              precision    recall  f1-score   support

 Anthracnose       0.78      0.86      0.82       230
      Canker       0.90      0.82      0.86       230
         Dot       0.55      0.54      0.54       230
     Healthy       0.82      0.92      0.87       230
        Rust       0.90      0.78      0.84       230

    accuracy                           0.78      1150
   macro avg       0.79      0.78      0.78      1150
weighted avg       0.79      0.78      0.78      1150



In [ ]:
## 10. Quadratic Discriminant Analysis (QDA)
print("\nTraining Quadratic Discriminant Analysis (QDA)...")
meta_model_qda = QuadraticDiscriminantAnalysis()
meta_model_qda.fit(X_meta_leaf, y_true)
y_meta_pred_qda = meta_model_qda.predict(X_meta_leaf)
print(f"QDA Accuracy: {accuracy_score(y_true, y_meta_pred_qda):.4f}")
print(classification_report(y_true, y_meta_pred_qda, target_names=leaf_classes))


Training Quadratic Discriminant Analysis (QDA)...
QDA Accuracy: 0.8122
              precision    recall  f1-score   support

 Anthracnose       0.79      0.90      0.84       230
      Canker       0.86      0.86      0.86       230
         Dot       0.64      0.55      0.59       230
     Healthy       0.82      0.90      0.86       230
        Rust       0.93      0.84      0.88       230

    accuracy                           0.81      1150
   macro avg       0.81      0.81      0.81      1150
weighted avg       0.81      0.81      0.81      1150



/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 2 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 3 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/p

In [ ]:
## 11. XGBoost Classifier
print("\nTraining XGBoost Classifier...")
meta_model_xgb = XGBClassifier()
meta_model_xgb.fit(X_meta_leaf, y_true)
y_meta_pred_xgb = meta_model_xgb.predict(X_meta_leaf)
print(f"XGBoost Accuracy: {accuracy_score(y_true, y_meta_pred_xgb):.4f}")
print(classification_report(y_true, y_meta_pred_xgb, target_names=leaf_classes))


Training XGBoost Classifier...
XGBoost Accuracy: 1.0000
              precision    recall  f1-score   support

 Anthracnose       1.00      1.00      1.00       230
      Canker       1.00      1.00      1.00       230
         Dot       1.00      1.00      1.00       230
     Healthy       1.00      1.00      1.00       230
        Rust       1.00      1.00      1.00       230

    accuracy                           1.00      1150
   macro avg       1.00      1.00      1.00      1150
weighted avg       1.00      1.00      1.00      1150



In [ ]:
## 12. CatBoost Classifier
print("\nTraining CatBoost Classifier...")
meta_model_cat = CatBoostClassifier(learning_rate=0.1, iterations=1000, depth=6)
meta_model_cat.fit(X_meta_leaf, y_true)
y_meta_pred_cat = meta_model_cat.predict(X_meta_leaf)
print(f"CatBoost Accuracy: {accuracy_score(y_true, y_meta_pred_cat):.4f}")
print(classification_report(y_true, y_meta_pred_cat, target_names=leaf_classes))


Training CatBoost Classifier...
0:	learn: 1.4154354	total: 77.3ms	remaining: 1m 17s
1:	learn: 1.2706946	total: 101ms	remaining: 50.1s
2:	learn: 1.1628695	total: 119ms	remaining: 39.6s
3:	learn: 1.0814496	total: 140ms	remaining: 35s
4:	learn: 1.0045287	total: 157ms	remaining: 31.3s
5:	learn: 0.9463506	total: 185ms	remaining: 30.6s
6:	learn: 0.8904729	total: 208ms	remaining: 29.6s
7:	learn: 0.8499197	total: 229ms	remaining: 28.4s
8:	learn: 0.8033879	total: 245ms	remaining: 27s
9:	learn: 0.7648264	total: 268ms	remaining: 26.6s
10:	learn: 0.7325591	total: 293ms	remaining: 26.4s
11:	learn: 0.7044228	total: 315ms	remaining: 26s
12:	learn: 0.6757468	total: 338ms	remaining: 25.6s
13:	learn: 0.6502387	total: 362ms	remaining: 25.5s
14:	learn: 0.6253692	total: 384ms	remaining: 25.2s
15:	learn: 0.6020617	total: 403ms	remaining: 24.8s
16:	learn: 0.5825740	total: 420ms	remaining: 24.3s
17:	learn: 0.5654387	total: 436ms	remaining: 23.8s
18:	learn: 0.5505957	total: 453ms	remaining: 23.4s
19:	learn: 0

In [ ]:
## 13. LightGBM Classifier
print("\nTraining LightGBM Classifier...")
meta_model_lgbm = LGBMClassifier()
meta_model_lgbm.fit(X_meta_leaf, y_true)
y_meta_pred_lgbm = meta_model_lgbm.predict(X_meta_leaf)
print(f"LightGBM Accuracy: {accuracy_score(y_true, y_meta_pred_lgbm):.4f}")
print(classification_report(y_true, y_meta_pred_lgbm, target_names=leaf_classes))


Training LightGBM Classifier...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000355 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 1150, number of used features: 10
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
